# Verification of simulation subpackage: Class to perform sensitivity analysis in qualities (QualitySensitivity)

The class will be used to compute a sensitivity analysis on an x-ray radiation quality.
It will be used to compute the effect on the spectrum and integral quantities due to changes in x-ray tube parameters, specifically the tube voltage, additional filtration thickness and additional filtration purity.

The class will, for a quality, a percentage deviation in a single tube parameter:
- Get the nominal parameters of the quality.
- Get the deviated parameters of the quality by applying the percentage deviation to the nominal parameters.
- Use the Quality class to simulate the spectrum and compute integral quantities for the nominal value and the perturbed value of the tube parameters.
- Compute the percentage difference in integral quantities.

Reference case for verification:
- N60 radiation quality.
- Anode angle of 20º.
- H*(10) operational quantity at 0º irradiation angle.
- mass energy transfer coefficients for air from PENELOPE 2018.
- air kerma to dose conversion coefficients from CMI 2025.
- measurement distance at 1 m.
- air thickness equal to distance.
- Deviation of 5% in tube voltage, additional filtration thickness and additional filtration purity (Pb).

In [1]:
import spekpy as sp
import numpy as np
from scipy.interpolate import Akima1DInterpolator
from metpyx.sim import QualitySensitivity

## Independent calculation

In [2]:
# Dictionary to store results
r_spek_cls = {
    'spectrum': {},
    'e_mean': {},
    'kerma': {},
    'hvl1_al': {},
    'hvl2_al': {},
    'hvl1_cu': {},
    'hvl2_cu': {},
    'hk_mean': {},
    'dose': {},
}

In [3]:
# Spek instance initialization
# Nominal
nominal = sp.Spek(kvp=60, th=20)
nominal.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 1000]])

# Perturbed high voltage
dev_hv = sp.Spek(kvp=60 * 1.05, th=20)
dev_hv.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 1000]])

# Perturbed additional filtration thickness
dev_ft = sp.Spek(kvp=60, th=20)
dev_ft.multi_filter([["Al", 4.0], ["Cu", 0.6 * 1.05], ["Air", 1000]])

# Perturbed additional filtration purity
pb_thick = 0.6 * (0.05 * 8.96) / (0.95 * 11.35)
dev_fp = sp.Spek(kvp=60, th=20)
dev_fp.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Pb", pb_thick], ["Air", 1000]])

In [4]:
# Get quality related integral quantities (not related to dose)
for name, spek in zip(['nominal', 'dev_hv', 'dev_ft', 'dev_fp'], [nominal, dev_hv, dev_ft, dev_fp]):
    r_spek_cls['spectrum'][name] = spek.get_spectrum(diff=False)
    r_spek_cls['e_mean'][name] = spek.get_emean()
    r_spek_cls['kerma'][name] = spek.get_kerma()
    r_spek_cls['hvl1_al'][name] = spek.get_hvl1()
    r_spek_cls['hvl2_al'][name] = spek.get_hvl2()
    r_spek_cls['hvl1_cu'][name] = spek.get_hvl1(matl="Cu")
    r_spek_cls['hvl2_cu'][name] = spek.get_hvl2(matl="Cu")

In [5]:
# Get mean conversion coefficient calculation

# Define mass energy transfer coefficients for air (PENELOPE 2018). Units (keV, cm²/g)
pene_2018_energies = [
    1.0, 1.1726, 1.25, 1.4, 1.5, 1.75, 2.0, 2.5, 3.0, 3.2063, 3.206301, 3.22391, 3.25051, 3.5, 3.61881, 4.0,
    5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 12.5, 14.0, 15.0, 17.5, 20.0, 25.0, 28.6633, 30.0, 35.0, 40.0, 50.0, 60.0,
    70.0, 80.0, 90.0, 100.0, 125.0, 140.0, 150.0, 175.0, 187.083, 200.0, 250.0, 300.0, 324.037, 350.0, 386.867,
    400.0, 474.342, 500.0, 574.456, 600.0, 673.537, 700.0, 800.0, 900.0, 1000.0, 1250.0, 1500.0, 1558.93,
    1750.0, 1870.83, 2000.0, 2345.21, 2500.0, 3000.0, 3240.37, 3500.0, 4000.0, 4500.0, 5000.0, 6000.0, 6480.74,
    7000.0, 8000.0, 9000.0
]
pene_2018_values = [
    3487.7, 2271.66, 1907.85, 1396.25, 1152.41, 746.85, 510.495, 267.712, 156.677, 128.597, 139.322, 139.22,
    136.749, 110.244, 99.9467, 74.3863, 38.3165, 22.1387, 13.8638, 9.20954762864289, 6.40746485225397,
    4.62692014996195, 2.30743153037744, 1.61505075253763, 1.30130979037554, 0.801145676760343,
    0.525990443101652, 0.262079759969996, 0.17213944380102, 0.150643910752649, 0.096166283094939,
    0.067006659158694, 0.040350724533314, 0.030060380204255, 0.025736225862943, 0.023919178948042,
    0.023273564441493, 0.023169137685326, 0.023905175732908, 0.024501233602389, 0.024926643691358,
    0.025877217107796, 0.026274062287156, 0.026655131026205, 0.027882238669658, 0.028683812435718,
    0.028972447929017, 0.029148771809045, 0.029415131704691, 0.029490249846129, 0.029698178702551,
    0.029685041261757, 0.029603764611649, 0.029549718949812, 0.029405101930449, 0.029197657857256,
    0.028890614121642, 0.028390536220701, 0.027936798166738, 0.026719405747041, 0.02562121786982,
    0.025359438553807, 0.02463108715039, 0.024240842040503, 0.023753744772526, 0.022671649894621,
    0.022309768351513, 0.021132688079239, 0.020639133403572, 0.020121558259145, 0.019347638398287,
    0.018700741656237, 0.018185053120065, 0.017326066208925, 0.016966042150857, 0.016690444038446,
    0.016199645451199, 0.015809252632524
]

# Define air-kerma-to-dose conversion coefficients for H*(10) at 0º irradiation angle (CMI 2025). Units (keV, Sv/Gy)
cmi_2025_energies_h_star_10_0 = [
    2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40,
    42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 65, 70, 75, 80, 85, 90, 95, 100, 110, 120, 130, 140, 150, 160, 170,
    180, 190, 200, 225, 240, 250, 275, 300, 325, 350, 375, 400, 425, 450, 500, 511, 550, 600, 662, 700, 800,
    900, 1000, 1117, 1173, 1250, 1332, 1500, 1700, 1750, 2000, 2400, 2500, 3000, 3500, 4000, 4440, 5000, 6000,
    6129, 7000, 8000, 9000, 10000, 12500, 15000, 17500, 20000, 25000, 30000, 35000, 40000, 45000, 50000
]
cmi_2025_values_h_star_10_0 = [
    0.0, 0.0, 0.0, 0.0, 0.0, 5.006211183e-07, 7.53411188163e-05, 0.0013399279666727, 0.008339974116065,
    0.027959365493142, 0.0649015031392173, 0.117892384153277, 0.182854496500547, 0.254785065497812,
    0.329684376352027, 0.404237490435794, 0.474883646496535, 0.543075936619701, 0.608764764690923,
    0.729121699136139, 0.834359989357258, 0.932217912526701, 1.019787991814, 1.10127304871649, 1.17923580150499,
    1.25093929821867, 1.31693499152686, 1.37836679008758, 1.43614832274801, 1.48895147535783, 1.53695987736966,
    1.579367978262, 1.61683555911064, 1.65111364109003, 1.67895036865187, 1.70147532324998, 1.71959205780654,
    1.73569389238546, 1.7513548512875, 1.7660601751939, 1.76411105061724, 1.75880008458157, 1.74401588142437,
    1.7259083944068, 1.7051120882291, 1.68285949300524, 1.660806895852, 1.62069771752482, 1.58655624975128,
    1.55253890207103, 1.5214025670048, 1.49292559713724, 1.47191391445754, 1.45095250563087, 1.43234388256437,
    1.41504458319957, 1.40045048056932, 1.36708755516683, 1.35062396617274, 1.34112411144134, 1.31829850613254,
    1.29920469418009, 1.28280551309179, 1.26779645957562, 1.25550014309865, 1.24492592905825, 1.23453216595746,
    1.225707899164, 1.2113467185313, 1.20789939579853, 1.19806539226821, 1.18818337262196, 1.17653859236809,
    1.17219827416785, 1.16041231542684, 1.14984262965137, 1.14263898967494, 1.13414788332302, 1.13119321285273,
    1.12823105732118, 1.12464830688625, 1.11896627040591, 1.11357509036097, 1.11227237543666, 1.1077193167192,
    1.10088553317439, 1.09989918128124, 1.09354396763082, 1.08802563926124, 1.08425634507853, 1.07995815186229,
    1.07712879613375, 1.0702298390959, 1.06941454173716, 1.06406552317947, 1.05908367593344, 1.05411440308156,
    1.05022231792777, 1.04082044834368, 1.03355131530843, 1.02708988121954, 1.02235595808979, 1.01449765630958,
    1.00889446838262, 1.00286377763245, 1.00129856074053, 0.997193631804752, 0.996021011235643
]

for name, spek in zip(['nominal', 'dev_hv', 'dev_ft', 'dev_fp'], [nominal, dev_hv, dev_ft, dev_fp]):

    # Get spectrum data
    energies = np.array(r_spek_cls['spectrum'][name][0])
    fluence = np.array(r_spek_cls['spectrum'][name][1])

    # Get mass energy transfer coefficients for air
    mu_energies = np.array(pene_2018_energies)
    mu_values = np.array(pene_2018_values)

    # Get air-kerma-to-dose conversion coefficients for H*(10) at 0º irradiation angle
    hk_energies = np.array(cmi_2025_energies_h_star_10_0)
    hk_values = np.array(cmi_2025_values_h_star_10_0)

    # If there are zeros in the values of mass energy transfer coefficients for air or
    # air-kerma-to-dose conversion coefficients, remove them to avoid log(0)
    if np.any(mu_values == 0):
        mask_mu = mu_values != 0
        filtered_mu_energies = mu_energies[mask_mu]
        filtered_mu_values = mu_values[mask_mu]
        print("Warning: Zeros found in mu_tr_over_rho_air values. They have been removed for interpolation.")
    else:
        filtered_mu_energies = mu_energies
        filtered_mu_values = mu_values

    if np.any(hk_values == 0):
        mask_hk = hk_values != 0
        filtered_hk_energies = hk_energies[mask_hk]
        filtered_hk_values = hk_values[mask_hk]
        print("Warning: Zeros found in h_k values. They have been removed for interpolation.")
    else:
        filtered_hk_energies = hk_energies
        filtered_hk_values = hk_values

    # Interpolating mass energy transfer coefficients for air coefficients to spectrum energies
    interpolator = Akima1DInterpolator(x=np.log(filtered_mu_energies), y=np.log(filtered_mu_values))
    interpolated_mu = np.exp(interpolator(np.log(energies)))

    # Interpolating air-kerma-to-dose conversion coefficients to spectrum energies
    interpolator = Akima1DInterpolator(x=np.log(filtered_hk_energies), y=np.log(filtered_hk_values))
    interpolated_hk = np.exp(interpolator(np.log(energies)))

    # Check for NaN values in interpolated results
    if np.any(np.isnan(interpolated_mu)):
        print("Warning: NaN values found in interpolated mu_tr_over_rho_air.")
    if np.any(np.isnan(interpolated_hk)):
        print("Warning: NaN values found in interpolated h_k.")

    # Calculating mean conversion coefficient (use np.nansum to ignore NaN values in the calculation)
    # Numerator: sum(phi(E) * E * mu_tr_over_rho(E) * h_K(E)):  keV * (1/cm²) * (cm²/g) * (Sv/Gy) = (keV/g) * (Sv/Gy)
    # Denominator: sum(phi(E) * E * mu_tr_over_rho(E)): keV * (1/cm²) * (cm²/g) = keV/g
    # Mean h_K: (keV/g) * (Sv/Gy) / (keV/g) = Sv/Gy
    hk_mean_numerator = np.nansum(fluence * energies * interpolated_mu * interpolated_hk)
    hk_mean_denominator = np.nansum(fluence * energies * interpolated_mu)
    hk_mean = hk_mean_numerator / hk_mean_denominator
    r_spek_cls['hk_mean'][name] = hk_mean

    # Get dose equivalent for H*(10) at 0º irradiation angle
    # Units: H = h_K * K_air: uGy * Sv/Gy = uSv
    r_spek_cls['dose'][name] = hk_mean * r_spek_cls['kerma'][name]

In [6]:
# Compute percentage deviation for each perturbed case and store in a dictionary
r_group = {
    'dev_hv': {},
    'dev_ft': {},
    'dev_fp': {},
}
for key2 in ['dev_hv', 'dev_ft', 'dev_fp']:
    for key1 in ['e_mean', 'kerma', 'hvl1_al', 'hvl2_al', 'hvl1_cu', 'hvl2_cu', 'hk_mean', 'dose']:
        nominal = r_spek_cls[key1]['nominal']
        perturbed = r_spek_cls[key1][key2]
        deviation = (perturbed - nominal) / nominal * 100
        r_group[key2][key1] = (nominal, perturbed, deviation)

## High voltage deviation

### Assertions

In [7]:
# N60 radiation quality, 5% deviation in tube voltage
q1 = QualitySensitivity("N60", 'tube_voltage', 5, th=20)

# Perturbation attributes
assert q1.quality == 'N60'
assert q1.parameter == 'tube_voltage'
assert q1.deviation == 5
assert q1.material is None
assert q1.distance == 100

# Nominal parameters
assert q1.nominal_params['tube_voltage'] == 60
assert q1.nominal_params['inherent_filtration'] == {'Al': 4}
assert q1.nominal_params['additional_filtration'] == {'Cu': 0.6}
assert q1.nominal_params['total_filtration'] == {'Al': 4, 'Cu': 0.6}
assert q1.nominal_params['spek_filtration'] == [["Al", 4.0], ["Cu", 0.6], ["Air", 1000]]

# Perturbed parameters
assert q1.perturbed_params['tube_voltage'] == 60 * 1.05  # +5%
assert q1.perturbed_params['inherent_filtration'] == {'Al': 4}  # No change
assert q1.perturbed_params['additional_filtration'] == {'Cu': 0.6}  # No change
assert q1.perturbed_params['total_filtration'] == {'Al': 4, 'Cu': 0.6}  # No change
assert q1.perturbed_params['spek_filtration'] == [["Al", 4.0], ["Cu", 0.6], ["Air", 1000]]  # No change

# Nominal spectrum
assert q1.nominal_spec.state.spectrum_parameters.z == 100
assert q1.nominal_spec.state.model_parameters.th == 20
assert q1.nominal_spec.state.model_parameters.kvp == 60
assert q1.nominal_spec.state.filtration.filters == [("Al", 4.0), ("Cu", 0.6), ("Air", 1000)]

# Perturbed spectrum
assert q1.perturbed_spec.state.spectrum_parameters.z == 100
assert q1.perturbed_spec.state.model_parameters.th == 20
assert q1.perturbed_spec.state.model_parameters.kvp == 60 * 1.05  # +5%
assert q1.perturbed_spec.state.filtration.filters == [("Al", 4.0), ("Cu", 0.6), ("Air", 1000)]

# Integral quantities (not related to dose): (nominal, perturbed, % difference)
assert q1.get_emean_dev() == r_group['dev_hv']['e_mean']
assert q1.get_kerma_dev() == r_group['dev_hv']['kerma']
assert q1.get_hvl1_dev() == r_group['dev_hv']['hvl1_al']
assert q1.get_hvl2_dev() == r_group['dev_hv']['hvl2_al']
assert q1.get_hvl1_dev(matl="Cu") == r_group['dev_hv']['hvl1_cu']
assert q1.get_hvl2_dev(matl="Cu") == r_group['dev_hv']['hvl2_cu']

# Integral quantities (related to dose): (nominal, perturbed, % difference)
assert q1.get_hk_mean_dev('h_star_10', 0) == r_group['dev_hv']['hk_mean']
assert q1.get_dose_equivalent_dev('h_star_10', 0) == r_group['dev_hv']['dose']

In [8]:
print("All assertions passed. Initializations are correct.")

All assertions passed. Initializations are correct.


## Additional filtration thickness deviation

### Assertions

In [9]:
# N60 radiation quality, 5% deviation in additional filtration thickness
q2 = QualitySensitivity("N60", 'additional_filtration_thickness', 5, th=20)

# Perturbation attributes
assert q2.quality == 'N60'
assert q2.parameter == 'additional_filtration_thickness'
assert q2.deviation == 5
assert q2.material is None
assert q2.distance == 100

# Nominal parameters
assert q2.nominal_params['tube_voltage'] == 60
assert q2.nominal_params['inherent_filtration'] == {'Al': 4}
assert q2.nominal_params['additional_filtration'] == {'Cu': 0.6}
assert q2.nominal_params['total_filtration'] == {'Al': 4, 'Cu': 0.6}
assert q2.nominal_params['spek_filtration'] == [["Al", 4.0], ["Cu", 0.6], ["Air", 1000]]

# Perturbed parameters
assert q2.perturbed_params['tube_voltage'] == 60  # No change
assert q2.perturbed_params['inherent_filtration'] == {'Al': 4}  # No change
assert q2.perturbed_params['additional_filtration'] == {'Cu': 0.6 * 1.05}  # +5%
assert q2.perturbed_params['total_filtration'] == {'Al': 4, 'Cu': 0.6 * 1.05}  # +5% in additional
assert q2.perturbed_params['spek_filtration'] == [["Al", 4.0], ["Cu", 0.6 * 1.05], ["Air", 1000]]  # +5% in additional

# Nominal spectrum
assert q2.nominal_spec.state.spectrum_parameters.z == 100
assert q2.nominal_spec.state.model_parameters.th == 20
assert q2.nominal_spec.state.model_parameters.kvp == 60
assert q2.nominal_spec.state.filtration.filters == [("Al", 4.0), ("Cu", 0.6), ("Air", 1000)]

# Perturbed spectrum
assert q2.perturbed_spec.state.spectrum_parameters.z == 100
assert q2.perturbed_spec.state.model_parameters.th == 20
assert q2.perturbed_spec.state.model_parameters.kvp == 60
assert q2.perturbed_spec.state.filtration.filters == [("Al", 4.0), ("Cu", 0.6 * 1.05), ("Air", 1000)]  # +5%

# Integral quantities (not related to dose): (nominal, perturbed, % difference)
assert q2.get_emean_dev() == r_group['dev_ft']['e_mean']
assert q2.get_kerma_dev() == r_group['dev_ft']['kerma'], f"Expected {r_group['dev_ft']['kerma']}, got {q2.get_kerma_dev()}"
assert q2.get_hvl1_dev() == r_group['dev_ft']['hvl1_al']
assert q2.get_hvl2_dev() == r_group['dev_ft']['hvl2_al']
assert q2.get_hvl1_dev(matl="Cu") == r_group['dev_ft']['hvl1_cu']
assert q2.get_hvl2_dev(matl="Cu") == r_group['dev_ft']['hvl2_cu']

# Integral quantities (related to dose): (nominal, perturbed, % difference)
assert q2.get_hk_mean_dev('h_star_10', 0) == r_group['dev_ft']['hk_mean']
assert q2.get_dose_equivalent_dev('h_star_10', 0) == r_group['dev_ft']['dose']

In [10]:
print("All assertions passed. Initializations are correct.")

All assertions passed. Initializations are correct.


## Additional filtration purity deviation

### Assertions

In [11]:
# N60 radiation quality, 5% deviation in additional filtration purity (Pb)
q3 = QualitySensitivity("N60", 'additional_filtration_purity', 5, material='Pb', th=20)

# Perturbation attributes
assert q3.quality == 'N60'
assert q3.parameter == 'additional_filtration_purity'
assert q3.deviation == 5
assert q3.material == 'Pb'
assert q3.distance == 100

pb_thick = 0.6 * (0.05 * 8.96) / (0.95 * 11.35)

# Nominal parameters
assert q3.nominal_params['tube_voltage'] == 60
assert q3.nominal_params['inherent_filtration'] == {'Al': 4}
assert q3.nominal_params['additional_filtration'] == {'Cu': 0.6}
assert q3.nominal_params['total_filtration'] == {'Al': 4, 'Cu': 0.6}
assert q3.nominal_params['spek_filtration'] == [["Al", 4.0], ["Cu", 0.6], ["Air", 1000]]

# Perturbed parameters
assert q3.perturbed_params['tube_voltage'] == 60  # No change
assert q3.perturbed_params['inherent_filtration'] == {'Al': 4}  # No change
assert q3.perturbed_params['additional_filtration'] == {'Cu': 0.6, 'Pb': pb_thick}  # +5%
assert q3.perturbed_params['total_filtration'] == {'Al': 4, 'Cu': 0.6, 'Pb': pb_thick}  # +5% in additional
assert q3.perturbed_params['spek_filtration'] == [["Al", 4.0], ["Cu", 0.6], ["Pb", pb_thick], ["Air", 1000]]  # +5% in additional

# Nominal spectrum
assert q3.nominal_spec.state.spectrum_parameters.z == 100
assert q3.nominal_spec.state.model_parameters.th == 20
assert q3.nominal_spec.state.model_parameters.kvp == 60
assert q3.nominal_spec.state.filtration.filters == [("Al", 4.0), ("Cu", 0.6), ("Air", 1000)]

# Perturbed spectrum
assert q3.perturbed_spec.state.spectrum_parameters.z == 100
assert q3.perturbed_spec.state.model_parameters.th == 20
assert q3.perturbed_spec.state.model_parameters.kvp == 60
assert q3.perturbed_spec.state.filtration.filters == [("Al", 4.0), ("Cu", 0.6), ("Pb", pb_thick), ("Air", 1000)]  # +5%

# Integral quantities (not related to dose): (nominal, perturbed, % difference)
assert q3.get_emean_dev() == r_group['dev_fp']['e_mean']
assert q3.get_kerma_dev() == r_group['dev_fp']['kerma']
assert q3.get_hvl1_dev() == r_group['dev_fp']['hvl1_al']
assert q3.get_hvl2_dev() == r_group['dev_fp']['hvl2_al']
assert q3.get_hvl1_dev(matl="Cu") == r_group['dev_fp']['hvl1_cu']
assert q3.get_hvl2_dev(matl="Cu") == r_group['dev_fp']['hvl2_cu']

# Integral quantities (related to dose): (nominal, perturbed, % difference)
assert q3.get_hk_mean_dev('h_star_10', 0) == r_group['dev_fp']['hk_mean']
assert q3.get_dose_equivalent_dev('h_star_10', 0) == r_group['dev_fp']['dose']

In [12]:
print("All assertions passed. Initializations are correct.")

All assertions passed. Initializations are correct.
